## Product query agent

In [28]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

## added the product

In [29]:
PRODUCTS = {
    "wireless headphones": {
        "price": 79.99,
        "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation.",
    },
    "smart watch": {
        "price": 199.99,
        "description": "Tracks heart rate and sleep. 5-day battery, water-resistant.",
    },
    "mechanical keyboard": {
        "price": 129.00,
        "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB.",
    },
    "laptop stand": {
        "price": 34.99,
        "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat.",
    },
}

REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch": {"reviews": 340, "rating": 3.9},
    "mechanical keyboard": {"reviews": 67, "rating": 4.8},
    "laptop stand": {"reviews": 781, "rating": 4.5},
}


@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)


@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return "Review not available for this product"
    return str(r)

In [30]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [31]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [32]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with a 30-hour battery life and active noise cancellation.


In [33]:
ask("what are the reviews on this product?")

I'm not able to directly provide reviews, but I can look up a product and provide its rating. If you provide the product name, I can assist you with that.


## in memory chat session 

In [34]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver(),
)


def ask2(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent2.invoke(
        {
            "messages": [{"role": "user", "content": question}],
        },
        config=config,
    )
    print(result["messages"][-1].content)

In [35]:
ask2("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [36]:
ask2("what are the reviews on this product?")

The wireless headphones have 1262 reviews with an average rating of 4.6.
